# Data Audit — Canadian Drug Shortage Prediction

Full audit of source data, intermediate models, and the mart before committing to survival modelling. Goal: surface any data assumptions that would change the modelling plan, and produce evidence for the writeup.

**Organized in four tiers**:

1. **Tier 1** — would change the modelling approach fundamentally (report semantics, observation universe, completeness over time)
2. **Tier 2** — would affect feature interpretation (DPD extract handling, discontinuation relationships, manufacturer identity)
3. **Tier 3** — quality checks (date fields, label integrity, feature sanity)
4. **Tier 4** — descriptive understanding for the writeup (class distribution, drug characteristics)

Each section ends with a **FINDING** cell stating a concrete one-line conclusion.

**External validation required**: a few questions about source data semantics can only be answered from the Health Canada / Drug Product Database data dictionaries. Those are flagged in the relevant sections — the audit doesn't depend on them but the final answers will be stronger with documentation checked.

## Setup

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
pd.set_option('display.max_rows', 100)

DB_PATH = Path.cwd().parent / "drug_shortages.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)

print('Connected. Schemas:')
con.sql("""
    SELECT table_schema, COUNT(*) AS n_tables
    FROM information_schema.tables
    WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
    GROUP BY table_schema
    ORDER BY table_schema
""").show()

# TIER 1 — would change modelling approach

## Q1. What does a 'shortage' mean in the source data?

Investigating whether reports are truly independent events or multi-part filings on a single disruption.

In [ ]:
# Q1a. Do report_ids ever appear more than once in the raw data?
# If yes, we have amendments that we may be collapsing silently.
con.sql('''
    SELECT
        COUNT(*) AS total_raw_rows,
        COUNT(DISTINCT "Report ID") AS distinct_report_ids,
        COUNT(*) - COUNT(DISTINCT "Report ID") AS duplicate_rows
    FROM raw.hc_shortages_raw
''').show()

In [ ]:
# Q1b. If there ARE duplicate report_ids, what differs between them?
# Pull a sample of duplicated rows to inspect.
con.sql('''
    WITH dups AS (
        SELECT "Report ID"
        FROM raw.hc_shortages_raw
        GROUP BY "Report ID"
        HAVING COUNT(*) > 1
        LIMIT 5
    )
    SELECT 
        r."Report ID", r."Shortage status", r."Actual start date", 
        r."Actual end date", r."Date Created", r."Date Updated",
        r.source_year_label
    FROM raw.hc_shortages_raw r
    INNER JOIN dups ON r."Report ID" = dups."Report ID"
    ORDER BY r."Report ID", r."Date Updated"
''').show()

In [ ]:
# Q1c. Same-DIN same-date shortages — do multiple reports get filed for the same event?
con.sql('''
    SELECT 
        COUNT(*) AS pairs_same_din_same_start_date,
        COUNT(DISTINCT din) AS unique_dins_affected
    FROM (
        SELECT din, actual_start_date
        FROM main_staging.stg_hc_shortages
        WHERE din IS NOT NULL AND actual_start_date IS NOT NULL
        GROUP BY din, actual_start_date
        HAVING COUNT(*) > 1
    )
''').show()

In [ ]:
# Q1d. Same-DIN overlapping shortages — check the time-overlap pattern.
# Two reports for the same DIN where one started before the other ended.
con.sql('''
    WITH shortage_pairs AS (
        SELECT 
            a.din,
            a.report_id AS report_a, b.report_id AS report_b,
            a.actual_start_date AS a_start, a.actual_end_date AS a_end,
            b.actual_start_date AS b_start, b.actual_end_date AS b_end
        FROM main_staging.stg_hc_shortages a
        INNER JOIN main_staging.stg_hc_shortages b 
            ON a.din = b.din AND a.report_id < b.report_id
        WHERE a.din IS NOT NULL 
          AND a.actual_start_date IS NOT NULL
          AND b.actual_start_date IS NOT NULL
          AND a.actual_end_date IS NOT NULL
          AND b.actual_start_date < a.actual_end_date
          AND b.actual_start_date > a.actual_start_date
    )
    SELECT 
        COUNT(*) AS overlapping_pairs,
        COUNT(DISTINCT din) AS dins_with_overlaps
    FROM shortage_pairs
''').show()

# Show a few examples
con.sql('''
    WITH shortage_pairs AS (
        SELECT 
            a.din,
            a.report_id AS report_a, b.report_id AS report_b,
            a.actual_start_date AS a_start, a.actual_end_date AS a_end,
            b.actual_start_date AS b_start, b.actual_end_date AS b_end,
            a.shortage_status AS a_status, b.shortage_status AS b_status
        FROM main_staging.stg_hc_shortages a
        INNER JOIN main_staging.stg_hc_shortages b 
            ON a.din = b.din AND a.report_id < b.report_id
        WHERE a.din IS NOT NULL 
          AND a.actual_start_date IS NOT NULL
          AND b.actual_start_date IS NOT NULL
          AND a.actual_end_date IS NOT NULL
          AND b.actual_start_date < a.actual_end_date
          AND b.actual_start_date > a.actual_start_date
    )
    SELECT * FROM shortage_pairs LIMIT 10
''').show()

**FINDING Q1**: 
- If `duplicate_rows > 0` in Q1a, reports are amended via multiple raw rows. We should investigate whether staging handles this correctly.
- If `pairs_same_din_same_start_date > 0` in Q1c, multiple reports can be filed for the same DIN starting the same day — possibly strength/form variants or multi-manufacturer reports.
- If `overlapping_pairs` in Q1d is large, concurrent reports on the same DIN exist and our episode-collapsing threshold needs to handle overlap, not just gap.

## Q2. How is the observation universe actually bounded?

Understanding which drugs are eligible to appear in the panel and whether there are drugs in shortages that we miss entirely.

In [ ]:
# Q2a. DINs in shortage data that DON'T appear in DPD. These are unmatched.
con.sql('''
    WITH shortage_dins AS (
        SELECT DISTINCT din 
        FROM main_staging.stg_hc_shortages 
        WHERE din IS NOT NULL
    ),
    dpd_dins AS (
        SELECT DISTINCT din 
        FROM main_intermediate.dim_drug_by_din
    )
    SELECT
        (SELECT COUNT(*) FROM shortage_dins) AS shortage_distinct_dins,
        (SELECT COUNT(*) FROM dpd_dins) AS dpd_distinct_dins,
        COUNT(*) AS unmatched_shortage_dins
    FROM shortage_dins s
    LEFT JOIN dpd_dins d ON s.din = d.din
    WHERE d.din IS NULL
''').show()

In [ ]:
# Q2b. How many shortage reports are from drugs NOT in DPD?
# And are those reports concentrated in specific years/categories?
con.sql('''
    SELECT
        COUNT(*) AS total_reports,
        COUNT(*) FILTER (WHERE has_din = FALSE) AS schedule_c_reports,
        COUNT(*) FILTER (
            WHERE has_din = TRUE 
              AND din NOT IN (SELECT din FROM main_intermediate.dim_drug_by_din)
        ) AS reports_with_din_but_not_in_dpd
    FROM main_staging.stg_hc_shortages
''').show()

In [ ]:
# Q2c. What do 'cancelled' DPD drugs look like? How much post-cancellation lag is there?
# A drug marked CANCELLED in DPD — do we still see shortage reports after the cancellation date?
con.sql('''
    WITH cancelled_drugs AS (
        SELECT din, current_status_since_date AS cancellation_date
        FROM main_intermediate.dim_drug_by_din
        WHERE current_status LIKE 'CANCELLED%'
          AND current_status_since_date IS NOT NULL
    )
    SELECT
        COUNT(*) AS cancelled_dins,
        COUNT(DISTINCT s.din) AS cancelled_dins_with_shortage_after,
        SUM(CASE WHEN s.actual_start_date > c.cancellation_date THEN 1 ELSE 0 END) AS post_cancellation_shortages,
        ROUND(AVG(CASE WHEN s.actual_start_date > c.cancellation_date 
                       THEN DATEDIFF('day', c.cancellation_date, s.actual_start_date)
                       END), 0) AS mean_days_post_cancellation
    FROM cancelled_drugs c
    LEFT JOIN main_staging.stg_hc_shortages s ON c.din = s.din
''').show()

**FINDING Q2**: 
- `unmatched_shortage_dins` shows how many real shortage reports we can't match to a DPD drug. Small = good.
- If `post_cancellation_shortages > 0`, our observation universe filter (cancelled = excluded) may be dropping real shortages. This is important for episode modelling.

## Q3. Completeness across time

Checking for reporting gaps, backfill patterns, and partial-month artifacts.

In [ ]:
# Q3a. Shortage start counts per month across the entire dataset.
monthly_shortages = con.sql('''
    SELECT 
        DATE_TRUNC('month', actual_start_date) AS start_month,
        COUNT(*) AS n_shortages
    FROM main_staging.stg_hc_shortages
    WHERE actual_start_date IS NOT NULL
    GROUP BY start_month
    ORDER BY start_month
''').df()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(monthly_shortages['start_month'], monthly_shortages['n_shortages'], marker='.')
ax.set_title('New shortage reports per month')
ax.set_ylabel('count')
ax.axvline(pd.Timestamp('2017-03-14'), color='red', ls='--', lw=0.8, label='Mandatory reporting begins')
ax.axvline(pd.Timestamp('2020-03-01'), color='orange', ls='--', lw=0.8, label='COVID onset')
ax.legend()
plt.tight_layout()
plt.show()

print('Last 6 months:')
print(monthly_shortages.tail(6).to_string(index=False))

In [ ]:
# Q3b. Backfill check — compare actual_start_date vs date_created.
# If reports are filed promptly, these should be close. If not, there's lag.
con.sql('''
    SELECT
        COUNT(*) AS n_reports,
        ROUND(AVG(DATEDIFF('day', actual_start_date, date_created)), 1) AS mean_lag_days,
        PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY DATEDIFF('day', actual_start_date, date_created)) AS median_lag_days,
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY DATEDIFF('day', actual_start_date, date_created)) AS p95_lag_days,
        COUNT(*) FILTER (WHERE date_created < actual_start_date) AS reports_created_before_start,
        COUNT(*) FILTER (WHERE date_created > actual_start_date + INTERVAL '180 days') AS reports_created_more_than_180d_later
    FROM main_staging.stg_hc_shortages
    WHERE actual_start_date IS NOT NULL
      AND date_created IS NOT NULL
''').show()

In [ ]:
# Q3c. Is 'recent' shortage data complete, or is there noticeable fall-off at the tail?
# If true mandatory reporting + 30-day deadline means end-of-data months should be smaller.
con.sql('''
    SELECT 
        DATE_TRUNC('month', actual_start_date) AS start_month,
        COUNT(*) AS n_shortages,
        COUNT(*) FILTER (WHERE date_created > actual_start_date + INTERVAL '30 days') AS filed_late,
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE date_created > actual_start_date + INTERVAL '30 days') 
                  / COUNT(*), 1
        ) AS pct_filed_late
    FROM main_staging.stg_hc_shortages
    WHERE actual_start_date IS NOT NULL AND date_created IS NOT NULL
      AND actual_start_date >= '2024-01-01'
    GROUP BY start_month
    ORDER BY start_month
''').show()

**FINDING Q3**: 
- Look at the chart for regime boundaries and any suspicious dropouts.
- `reports_created_before_start` shows anticipatory filings (shortage disclosed before starting) — important because our current exclusion is based on `actual_start_date`.
- `pct_filed_late` in recent months tells us the lag pattern and whether the tail is underreported.

# TIER 2 — would affect feature interpretation

## Q4. DPD multi-extract handling

In [ ]:
# Q4a. How many DINs appear in multiple status extracts?
# A DIN in 'marketed' and also 'cancelled' means it has two drug_codes.
con.sql('''
    WITH din_extracts AS (
        SELECT 
            din,
            COUNT(DISTINCT product_status_extract) AS n_extracts,
            COUNT(DISTINCT drug_code) AS n_drug_codes
        FROM main_staging.stg_dpd_drug
        WHERE din IS NOT NULL
        GROUP BY din
    )
    SELECT
        COUNT(*) AS total_dins,
        COUNT(*) FILTER (WHERE n_extracts = 1) AS single_extract,
        COUNT(*) FILTER (WHERE n_extracts = 2) AS two_extracts,
        COUNT(*) FILTER (WHERE n_extracts = 3) AS three_extracts,
        COUNT(*) FILTER (WHERE n_extracts = 4) AS four_extracts,
        COUNT(*) FILTER (WHERE n_drug_codes > n_extracts) AS more_codes_than_extracts
    FROM din_extracts
''').show()

In [ ]:
# Q4b. For DINs in multiple extracts, which combinations are most common?
con.sql('''
    WITH din_extracts AS (
        SELECT 
            din,
            STRING_AGG(DISTINCT product_status_extract, ',' ORDER BY product_status_extract) AS extract_combo
        FROM main_staging.stg_dpd_drug
        WHERE din IS NOT NULL
        GROUP BY din
    )
    SELECT extract_combo, COUNT(*) AS n_dins
    FROM din_extracts
    GROUP BY extract_combo
    ORDER BY n_dins DESC
''').show()

In [ ]:
# Q4c. Does ai_group_no vary across extracts for the same DIN?
# Our drug_to_ai_group CTE assumes consistency via preference ordering.
con.sql('''
    WITH din_ai_groups AS (
        SELECT 
            d.din,
            COUNT(DISTINCT d.ai_group_no) AS n_distinct_ai_groups
        FROM main_staging.stg_dpd_drug d
        WHERE d.din IS NOT NULL
          AND d.ai_group_no IS NOT NULL
        GROUP BY d.din
        HAVING COUNT(DISTINCT d.ai_group_no) > 1
    )
    SELECT 
        COUNT(*) AS dins_with_inconsistent_ai_group,
        MAX(n_distinct_ai_groups) AS max_distinct_groups_per_din
    FROM din_ai_groups
''').show()

**FINDING Q4**: 
- If significant DINs appear in 2+ extracts, the preference ordering in `dim_drug_by_din` matters.
- If `dins_with_inconsistent_ai_group > 0`, our molecule-level peer features may partially fragment.
- The extract_combo breakdown tells us which transitions are most common in practice.

## Q5. The discontinuation data

In [ ]:
# Q5a. Basic shape of discontinuations and date coverage.
con.sql('''
    SELECT
        COUNT(*) AS total_discontinuations,
        COUNT(DISTINCT din) AS unique_dins,
        MIN(discontinuation_date) AS min_date,
        MAX(discontinuation_date) AS max_date,
        COUNT(*) FILTER (WHERE has_din = FALSE) AS schedule_c_discontinuations,
        COUNT(*) FILTER (WHERE is_tier_3 = TRUE) AS tier_3_discontinuations
    FROM main_staging.stg_hc_discontinuations
''').show()

In [ ]:
# Q5b. How does discontinuation data relate to DPD cancellation status?
# If a DIN is discontinued in HC data, is it also marked cancelled in DPD?
con.sql('''
    WITH disc_dins AS (
        SELECT DISTINCT din 
        FROM main_staging.stg_hc_discontinuations
        WHERE has_din = TRUE
    )
    SELECT
        (SELECT COUNT(*) FROM disc_dins) AS dins_with_disc_report,
        COUNT(*) FILTER (WHERE d.current_status LIKE 'CANCELLED%') AS also_cancelled_in_dpd,
        COUNT(*) FILTER (WHERE d.current_status LIKE 'MARKETED%') AS still_marketed_in_dpd,
        COUNT(*) FILTER (WHERE d.current_status IS NULL) AS not_in_dpd
    FROM disc_dins dd
    LEFT JOIN main_intermediate.dim_drug_by_din d ON dd.din = d.din
''').show()

In [ ]:
# Q5c. Are there DINs with BOTH shortage and discontinuation reports?
# And timing relationship — do shortages precede discontinuations?
con.sql('''
    WITH drug_history AS (
        SELECT
            s.din,
            MIN(s.actual_start_date) AS first_shortage,
            MAX(s.actual_start_date) AS last_shortage,
            MIN(d.discontinuation_date) AS first_discontinuation
        FROM main_staging.stg_hc_shortages s
        INNER JOIN main_staging.stg_hc_discontinuations d ON s.din = d.din
        WHERE s.din IS NOT NULL
        GROUP BY s.din
    )
    SELECT
        COUNT(*) AS dins_with_both,
        COUNT(*) FILTER (WHERE first_shortage < first_discontinuation) AS shortage_before_discontinuation,
        COUNT(*) FILTER (WHERE first_shortage > first_discontinuation) AS discontinuation_before_shortage,
        ROUND(AVG(DATEDIFF('day', first_shortage, first_discontinuation)), 0) AS mean_days_shortage_to_disc
    FROM drug_history
''').show()

**FINDING Q5**: 
- Total discontinuations, tier-3 counts, and DIN coverage.
- Relationship to DPD cancellation status shows whether signal is redundant.
- Timing pattern tells us if shortages precede discontinuations (consistent with supply-failure→exit hypothesis).

## Q6. Manufacturer identity stability

In [ ]:
# Q6a. Is company_code stably 1:1 with company_name?
con.sql('''
    WITH code_name_pairs AS (
        SELECT DISTINCT company_code, UPPER(TRIM(company_name)) AS company_name
        FROM main_staging.stg_dpd_comp
        WHERE company_code IS NOT NULL AND company_name IS NOT NULL
    )
    SELECT
        (SELECT COUNT(*) FROM code_name_pairs) AS distinct_code_name_pairs,
        (SELECT COUNT(DISTINCT company_code) FROM code_name_pairs) AS distinct_codes,
        (SELECT COUNT(DISTINCT company_name) FROM code_name_pairs) AS distinct_names
''').show()

In [ ]:
# Q6b. Company codes with more than one name (e.g., name changes, typos, or re-assignments).
con.sql('''
    WITH code_name_counts AS (
        SELECT company_code, COUNT(DISTINCT UPPER(TRIM(company_name))) AS n_names
        FROM main_staging.stg_dpd_comp
        WHERE company_code IS NOT NULL
        GROUP BY company_code
    )
    SELECT
        COUNT(*) AS total_codes,
        COUNT(*) FILTER (WHERE n_names = 1) AS codes_with_one_name,
        COUNT(*) FILTER (WHERE n_names > 1) AS codes_with_multiple_names,
        MAX(n_names) AS max_names_per_code
    FROM code_name_counts
''').show()

In [ ]:
# Q6c. Sample of company codes with multiple names — what do they look like?
con.sql('''
    WITH code_name_counts AS (
        SELECT company_code, COUNT(DISTINCT UPPER(TRIM(company_name))) AS n_names
        FROM main_staging.stg_dpd_comp
        WHERE company_code IS NOT NULL
        GROUP BY company_code
        HAVING COUNT(DISTINCT UPPER(TRIM(company_name))) > 1
    )
    SELECT DISTINCT c.company_code, UPPER(TRIM(c.company_name)) AS company_name
    FROM main_staging.stg_dpd_comp c
    INNER JOIN code_name_counts cnc ON c.company_code = cnc.company_code
    ORDER BY c.company_code, company_name
    LIMIT 30
''').show()

**FINDING Q6**: 
- If distinct_codes ≠ distinct_names, some companies have variant names under the same code (OK, model is fine).
- If `codes_with_multiple_names > 0`, check sample — minor spelling variants are OK, major changes suggest real entity changes that could fragment our manufacturer features.

# TIER 3 — quality checks

## Q7. Date field semantics

In [ ]:
# Q7a. Null patterns in the shortage date fields.
con.sql('''
    SELECT
        COUNT(*) AS total_reports,
        COUNT(anticipated_start_date) AS have_anticipated_start,
        COUNT(actual_start_date)      AS have_actual_start,
        COUNT(estimated_end_date)     AS have_estimated_end,
        COUNT(actual_end_date)        AS have_actual_end,
        COUNT(date_created)           AS have_date_created,
        COUNT(date_updated)           AS have_date_updated,
        COUNT(*) FILTER (
            WHERE actual_start_date IS NULL AND anticipated_start_date IS NOT NULL
        ) AS anticipated_only_no_actual
    FROM main_staging.stg_hc_shortages
''').show()

In [ ]:
# Q7b. Future-dated reports — actual_start_date in the future relative to date_created.
con.sql('''
    SELECT
        COUNT(*) AS future_dated_reports,
        COUNT(*) FILTER (WHERE DATEDIFF('day', date_created, actual_start_date) > 90) AS future_by_more_than_90d
    FROM main_staging.stg_hc_shortages
    WHERE actual_start_date IS NOT NULL
      AND date_created IS NOT NULL
      AND actual_start_date > date_created
''').show()

In [ ]:
# Q7c. actual_end_date BEFORE actual_start_date — nonsense rows.
con.sql('''
    SELECT
        COUNT(*) AS reports_with_inverted_dates,
        COUNT(*) FILTER (WHERE shortage_status = 'Resolved') AS inverted_and_resolved,
        COUNT(*) FILTER (WHERE shortage_status = 'Avoided shortage') AS inverted_and_avoided
    FROM main_staging.stg_hc_shortages
    WHERE actual_start_date IS NOT NULL
      AND actual_end_date IS NOT NULL
      AND actual_end_date < actual_start_date
''').show()

**FINDING Q7**: 
- Null patterns tell us which dates we can always rely on.
- Future-dated reports exist — these are anticipatory filings.
- Inverted dates indicate data-quality issues that our `has_suspect_dates` flag should be catching.

## Q8. Label quality

In [ ]:
# Q8a. Distribution of lifetime shortages per DIN in the panel.
lifetime_shortages = con.sql('''
    SELECT 
        din, 
        COUNT(*) AS lifetime_shortages
    FROM main_intermediate.fct_shortage_episode
    WHERE din IS NOT NULL
    GROUP BY din
''').df()

print('Distribution of lifetime shortages per DIN (among DINs with any):')
print(lifetime_shortages['lifetime_shortages'].describe())
print()
print('Quantiles:')
for q in [0.50, 0.75, 0.90, 0.95, 0.99, 1.00]:
    print(f'  p{int(q*100)}: {lifetime_shortages["lifetime_shortages"].quantile(q):.0f}')

In [ ]:
# Q8b. Label violations — rows flagged as was_in_shortage_on_obs_date=TRUE that shouldn't be eligible.
# Also: is shortage_started_within_90d truly always 0 when was_in_shortage_on_obs_date is 1?
con.sql('''
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE was_in_shortage_on_obs_date = 1 AND shortage_started_within_90d = 1
        ) AS in_shortage_and_another_starting,
        COUNT(*) FILTER (
            WHERE was_in_shortage_on_obs_date = 0 AND shortage_started_within_90d = 1
        ) AS true_positives,
        COUNT(*) FILTER (
            WHERE was_in_shortage_on_obs_date = 1
        ) AS in_shortage_rows
    FROM main_marts.mrt_shortage_panel
''').show()

In [ ]:
# Q8c. What fraction of all positive labels come from the top-N most shortage-prone DINs?
# This tells us if the 'predictability' is concentrated on a small set of problem drugs.
con.sql('''
    WITH din_positives AS (
        SELECT din, SUM(shortage_started_within_90d) AS n_positives
        FROM main_marts.mrt_shortage_panel
        WHERE was_in_shortage_on_obs_date = 0
        GROUP BY din
    ),
    ranked AS (
        SELECT din, n_positives, 
               ROW_NUMBER() OVER (ORDER BY n_positives DESC) AS rk,
               SUM(n_positives) OVER () AS total_positives,
               COUNT(*) OVER () AS total_dins
        FROM din_positives
        WHERE n_positives > 0
    )
    SELECT
        SUM(CASE WHEN rk <= 10     THEN n_positives ELSE 0 END) * 100.0 / MAX(total_positives) AS pct_top_10_dins,
        SUM(CASE WHEN rk <= 50     THEN n_positives ELSE 0 END) * 100.0 / MAX(total_positives) AS pct_top_50_dins,
        SUM(CASE WHEN rk <= 100    THEN n_positives ELSE 0 END) * 100.0 / MAX(total_positives) AS pct_top_100_dins,
        SUM(CASE WHEN rk <= 500    THEN n_positives ELSE 0 END) * 100.0 / MAX(total_positives) AS pct_top_500_dins,
        MAX(total_dins) AS total_dins_with_any_positive,
        MAX(total_positives) AS total_positives
    FROM ranked
''').show()

**FINDING Q8**: 
- `in_shortage_and_another_starting` should be 0 or near-zero by construction. If not, the exclusion logic is wrong.
- Lifetime-shortage distribution tells us if the signal is heavy-tailed.
- Top-N concentration tells us whether the model is effectively memorizing a small set of perpetually-problematic drugs.

## Q9. Feature sanity at extremes

In [ ]:
# Q9a. Top mfr_shortage_rate_12m values — are these real or artifacts?
con.sql('''
    SELECT 
        mfr_shortage_rate_12m,
        mfr_shortages_prior_12m,
        mfr_portfolio_size,
        COUNT(*) AS n_rows
    FROM main_marts.mrt_shortage_panel
    WHERE mfr_shortage_rate_12m > 0.5
    GROUP BY 1,2,3
    ORDER BY mfr_shortage_rate_12m DESC
    LIMIT 10
''').show()

In [ ]:
# Q9b. Extreme drug_age values — 50+ years is plausible, 100+ might be wrong.
con.sql('''
    SELECT
        COUNT(*) FILTER (WHERE drug_age_years > 50)  AS older_than_50y,
        COUNT(*) FILTER (WHERE drug_age_years > 100) AS older_than_100y,
        MAX(drug_age_years) AS max_age_years
    FROM main_marts.mrt_shortage_panel
''').show()

In [ ]:
# Q9c. DINs where shortages_all_prior is unusually high.
con.sql('''
    SELECT din, MAX(shortages_all_prior) AS max_lifetime_shortages_at_obs
    FROM main_marts.mrt_shortage_panel
    GROUP BY din
    ORDER BY max_lifetime_shortages_at_obs DESC
    LIMIT 10
''').show()

**FINDING Q9**: 
- Extreme mfr_shortage_rate_12m values should be investigated — a rate above 1.0 would mean shortages > portfolio size (possible if shortages include now-discontinued DINs) and deserves explanation.
- Extreme drug ages should correspond to legitimate historical drugs (aspirin, etc.).

# TIER 4 — descriptive understanding

## Q10. The positive class

In [ ]:
# Q10. Per-DIN positive label counts — full distribution.
din_stats = con.sql('''
    SELECT 
        din,
        COUNT(*) AS n_obs,
        SUM(shortage_started_within_90d) AS n_positives,
        SUM(was_in_shortage_on_obs_date) AS n_in_shortage
    FROM main_marts.mrt_shortage_panel
    GROUP BY din
''').df()

print(f'Total DINs in panel: {len(din_stats):,}')
print(f'DINs with zero positives: {(din_stats["n_positives"] == 0).sum():,} '
      f'({(din_stats["n_positives"] == 0).mean():.1%})')
print(f'DINs with zero in-shortage observations: {(din_stats["n_in_shortage"] == 0).sum():,}')
print()
print('Among DINs with at least one positive:')
positive_dins = din_stats[din_stats['n_positives'] > 0]
print(positive_dins['n_positives'].describe().round(2))

**FINDING Q10**: 
- Fraction of DINs that never have a positive label.
- Distribution of positives per DIN among those that have any.
- High concentration among repeat-offender DINs is consistent with the ML=heuristic result we already observed.

## Q11. Drug product characteristics

In [ ]:
# Q11a. Ingredient count distribution — how many combination drugs?
con.sql('''
    SELECT 
        ingredient_count,
        COUNT(DISTINCT din) AS n_dins
    FROM main_marts.mrt_shortage_panel
    GROUP BY ingredient_count
    ORDER BY ingredient_count
''').show()

In [ ]:
# Q11b. Coverage of ATC classification.
con.sql('''
    SELECT
        COUNT(DISTINCT din) AS total_dins_in_panel,
        COUNT(DISTINCT din) FILTER (WHERE has_atc_classification = TRUE) AS with_atc,
        COUNT(DISTINCT din) FILTER (WHERE has_atc_classification = FALSE) AS without_atc
    FROM main_marts.mrt_shortage_panel
''').show()

In [ ]:
# Q11c. Top pharmaceutical forms and routes.
print('Top forms:')
con.sql('''
    SELECT primary_form, COUNT(DISTINCT din) AS n_dins
    FROM main_marts.mrt_shortage_panel
    WHERE primary_form IS NOT NULL
    GROUP BY primary_form
    ORDER BY n_dins DESC
    LIMIT 10
''').show()

print('Top routes:')
con.sql('''
    SELECT primary_route, COUNT(DISTINCT din) AS n_dins
    FROM main_marts.mrt_shortage_panel
    WHERE primary_route IS NOT NULL
    GROUP BY primary_route
    ORDER BY n_dins DESC
    LIMIT 10
''').show()

**FINDING Q11**: Useful descriptive context for the writeup — what kinds of drugs dominate the panel, what fraction are combinations, ATC coverage.

## Final summary

Fill this in after running all sections. Each row is a concrete finding that affects either the plan or the writeup.

| Q | Finding | Action |
|---|---|---|
| Q1 | Report semantics | |
| Q2 | Observation universe | |
| Q3 | Temporal completeness | |
| Q4 | DPD extract handling | |
| Q5 | Discontinuations | |
| Q6 | Manufacturer identity | |
| Q7 | Date field semantics | |
| Q8 | Label quality | |
| Q9 | Feature extremes | |
| Q10 | Positive class distribution | |
| Q11 | Drug characteristics | |

**External documentation to verify separately:**

- Health Canada drug shortages data dictionary: how are report amendments handled? Are new report_ids issued for updates, or the same report_id with new date_updated?
- Drug Product Database: what does 'DORMANT' status mean vs 'CANCELLED'?
- Mandatory reporting rules: what's the 30-day filing deadline exactly? Does it apply to all shortages or just tier-3?

In [ ]:
con.close()